In [51]:
import numpy as np
from matplotlib import pyplot as plt
import numba
import time
get_ipython().run_line_magic('matplotlib', 'auto')

Using matplotlib backend: MacOSX


In [52]:
plt.rcParams.update({'font.size': 17}) # keep those graph fonts readable!
plt.rcParams['figure.dpi'] = 120



def init(xmax):
    plt.xlim((0, xmax-1))
    plt.grid('on')
    ax.set_xlabel('Grid Cells ($z$)')
    ax.set_ylabel('$E_x (eV^2)$')
    plt.show()

def graph(t):
    plt.clf()
    ax = fig.add_axes([.25, .25, .6, .6])
    
    img = ax.contourf(Ez)
    cbar=plt.colorbar(img, ax=ax)
    cbar.set_label('$E_z$ (arb.units)')
    ax.set_title('frame time{}'.format(t))
    plt.savefig('/Users/szechingaudreyfung/Desktop/PHYS 879 HPC/Projects/plots/planeNoAxion/frame{}.png'.format(t))
    plt.show()
    plt.pause(0.01)

Everything here is in natural units

# 2D

## pulse

In [53]:
nsteps = 2000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*np.pi*0.01)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.show()
print(t0, spread)

360 60


In [54]:
@numba.jit(nopython=True)
def Eupdate2d(Ez, Bx, By):
    for y in range(1,kmax-1):
        for x in range(1,kmax-1):
            Ez[y,x] = Ez[y,x] + 0.5*(By[y,x] - By[y,x-1] - Bx[y,x]+ Bx[y-1,x])
    return Ez

@numba.jit(nopython=True)
def Bupdate2d(Ez, Bx, By):
    for y in range(0,kmax-1):
        for x in range(0,kmax-1):
            Bx[y,x] = Bx[y,x] + 0.5*(Ez[y,x] - Ez[y+1,x])
            By[y,x] = By[y,x] + 0.5*(Ez[y,x+1] - Ez[y,x])
            
    return Bx, By


In [13]:
nsteps = 1000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

kmax = 1000
Ez = np.zeros([kmax, kmax])
Bx = np.zeros([kmax, kmax])
By = np.zeros([kmax, kmax])

# source
isource = int(kmax/2)
jsource = int(kmax/2)

cycle = 100
fig = plt.figure(figsize=(8,6))
start = time.time()

for t in range(nsteps+1):
    pulse = get_source(t)
    
    # ABC (doesn't work)
    Ez_past = Ez
    
    Ez = Eupdate2d(Ez, Bx, By)
    Ez[jsource, isource] = Ez[jsource, isource] + pulse
    
    #ABC (doesn't work)
    Ez[1:,0] = Ez_past[1:,1] - 1/3* (Ez[1:,1] - Ez_past[1:,0]) #\
    #- 1/3 * (Bx[1:,0] - Bx[:-1,0] +Bx[1:,1] - Bx[:-1,1])
    
    Ez[1:,-1] = Ez_past[1:,-2] - 1/3 * (Ez[1:,-2] - Ez_past[1:,-1]) #\
    #- 1/3 * (Bx[1:,-1] - Bx[:-1,-1]+ Bx[1:,-2] - Bx[:-1,-2])
    
    Ez[0, 1:] = Ez_past[1,1:] - 1/3 * (Ez[1,1:] - Ez_past[0,1:])# \
    #- 1/3 * (By[0,1:] - By[0,:-1]+By[1,1:]-By[1,:-1])
    
    Ez[-1, 1:] = Ez_past[-2,1:] - 1/3* (Ez[-2,1:] - Ez_past[-1,1:])# \
    #- 1/3 * (By[-1,1:] - By[-1,:-1] + By[-2,1:] - By[-2,:-1])

    
    Bx, By = Bupdate2d(Ez, Bx, By)
    
    if t%cycle == 0:
        graph(t)
        
print(time.time()-start)

7.549415111541748


### 1D plot

In [67]:
plt.clf()
plt.plot(np.arange(kmax), Ez[:,int(kmax/2)])
plt.show()

## soft source

In [24]:
def get_source(t):
    t0 = nsteps/5
    spread = nsteps/30
    source = np.exp(-0.5*(t-t0)**2/spread**2)
    return source
nsteps = 1000
plt.clf()
plt.plot(np.arange(nsteps), get_source(np.arange(nsteps)))
plt.xlim(0,nsteps)
plt.ylim(-1,1)
plt.show()

In [25]:
@numba.jit(nopython=True)
def Eupdate2d(Ez, Bx, By):
    for y in range(1,kmax-1):
        for x in range(1,kmax-1):
            Ez[y,x] = Ez[y,x] + 0.5*(By[y,x] - By[y,x-1] - Bx[y,x]+ Bx[y-1,x])
    return Ez

@numba.jit(nopython=True)
def Bupdate2d(Ez, Bx, By):
    for y in range(0,kmax-1):
        for x in range(0,kmax-1):
            Bx[y,x] = Bx[y,x] + 0.5*(Ez[y,x] - Ez[y+1,x])
            By[y,x] = By[y,x] + 0.5*(Ez[y,x+1] - Ez[y,x])
            
    return Bx, By

In [26]:
nsteps = 1000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

kmax = 1000
Ez = np.zeros([kmax, kmax])
Bx = np.zeros([kmax, kmax])
By = np.zeros([kmax, kmax])

# source
isource = int(kmax/2)
jsource = int(kmax/2)

cycle = 100
fig = plt.figure(figsize=(8,6))

for t in range(nsteps+1):
    pulse = get_source(t)
    
    Ez = Eupdate2d(Ez, Bx, By)
    Ez[jsource, isource] = Ez[jsource, isource] + pulse
    
    Bx, By = Bupdate2d(Ez, Bx, By)
    
    if t%cycle == 0:
        graph(t)

## plane waves

In [56]:
@numba.jit(nopython=True)
def get_source(t):
    E0 = 0.5
    wavelength=800
    t0 = 0
    #w = 2*np.pi/(wavelength)
    w = 0.002*np.pi
    #mask = (t>t0)*1
    source = E0*np.sin(w*t)#*mask
    return source

plt.clf()
spread = 60
t0 = spread*6
nsteps=1000
plt.plot(np.arange(0,nsteps), get_source(np.arange(0,nsteps)), label='source')
plt.ylim(-1,1)
plt.legend()
plt.show()

In [57]:
@numba.jit(nopython=True)
def Eupdate2d(Ez, Bx, By):
    for y in range(1,kmax-1):
        for x in range(1,kmax-1):
            Ez[y,x] = Ez[y,x] + 0.5*(By[y,x] - By[y,x-1] - Bx[y,x]+ Bx[y-1,x])
    return Ez

@numba.jit(nopython=True)
def Bupdate2d(Ez, Bx, By):
    for y in range(0,kmax-1):
        for x in range(0,kmax-1):
            Bx[y,x] = Bx[y,x] + 0.5*(Ez[y,x] - Ez[y+1,x])
            By[y,x] = By[y,x] + 0.5*(Ez[y,x+1] - Ez[y,x])
            
    return Bx, By

In [58]:
nsteps = 1000
t = np.arange(0,nsteps+1)
spread = 60
t0 = spread*6

kmax = 2000
Ez = np.zeros([kmax, kmax])
Bx = np.zeros([kmax, kmax])
By = np.zeros([kmax, kmax])

# source
isource = int(kmax/2)
jsource = int(kmax/2)

plot1d=True

if plot1d==False:
    cycle = 100
    fig = plt.figure(figsize=(8,6))

if plot1d == True:
    plt.clf()
    plt.close()
    cycle = 100
    lw=2
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_axes([.18, .18, .7, .7])
    xrange = np.linspace(0,kmax, kmax)
    [im] = ax.plot(xrange,Ez[int(kmax/2),:],linewidth=lw)
    [im2] = ax.plot(xrange,By[int(kmax/2),:],linewidth=lw)
    [im3] = ax.plot(xrange,Bx[int(kmax/2),:],linewidth=lw)
    im.set_color('orange')
    im2.set_color('blue')
    im3.set_color('red')
    init(kmax)
    plt.legend(['Ez', 'By', 'Bx'])
    plt.ylim(-1, 1)



for t in range(nsteps+1):
    pulse = get_source(t)
    
    Ez = Eupdate2d(Ez, Bx, By)
    Ez[:, isource] = Ez[:, isource] + pulse
    
    Bx, By = Bupdate2d(Ez, Bx, By)
    
    if t%cycle == 0:
        if plot1d== False:
            graph(t)
        if plot1d == True:
            im.set_ydata(Ez[int(kmax/2),:]) # orange
            im2.set_ydata(By[int(kmax/2),:]) # blue
            im3.set_ydata(Bx[int(kmax/2),:]) # red
            ax.set_title("frame time {}".format(t))
            plt.savefig('/Users/szechingaudreyfung/Desktop/PHYS 879 HPC/Projects/plots/planeNoAxion1d/NoAx1d{}.png'.format(t))
            plt.show()
            plt.pause(0.05)